# A top-down quantile model against the bottom-up bands

> **Everything below is fitted to simulated data, not real telemetry.** The "history" is the
> model's own simulated weeks (`evidence_kind="illustrative_synthetic"`), so the comparison tests
> two *estimators* on the same synthetic world. It says nothing about how either would do on a
> real fleet.

**Question.** A supplier's data scientist has no physics model of the fleet, only fleet-level
history: how many kW of turn-down the fleet could deliver in each half-hour, with calendar and
weather beside it. Could a plain machine-learning quantile model, fitted on that history alone,
forecast tomorrow's deliverable MW as well as the bottom-up bands the model builds EV by EV?

- **Bottom-up** (the model as built, trading contract §10.2–§10.3): each simulated week is an
  exchangeable draw of what a night could be, so the day-ahead band for a half-hour is the spread
  of the fleet deliverable across the weeks at that half-hour. The intraday band
  (§10.2d) re-forecasts the rest of the night at 17:00 from the EVs already on the driveway.
- **Top-down** (this notebook): scikit-learn's `GradientBoostingRegressor` with the quantile
  (pinball) loss, one model per level, trained on features known at the day-ahead decision, then
  widened by a split-conformal correction (conformalised quantile regression, CQR; Romano,
  Patterson and Candès 2019) so its P10–P90 band has close to 80% coverage by construction.
  It is fitted twice: without and with one price feature, the rank of each half-hour's price in
  the night as the planner could see it at the day-ahead decision.

The 300-EV default fleet and the visible price-rank feature follow the lead's ruling on this
notebook's first review (overnight review log, 30 September 2026).

Both are scored on held-out simulated weeks that neither saw: coverage of the P10–P90 band,
pinball loss at P10, P50 and P90, and sharpness (the mean band width).

**Sections**

1. The simulated history and its features (and the information cut-off)
2. Leave-weeks-out split
3. The top-down model: gradient-boosted quantiles
4. The conformal correction
5. Held-out comparison: coverage, pinball loss, sharpness
6. One test week, drawn
7. Reliability
8. Firm share against fleet size
9. Sanity checks
10. Limitations

In [1]:
# --- IMPORTS (one cell) -------------------------------------------------------
import time
from datetime import date

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import sklearn
from scipy.stats import rankdata
from sklearn.ensemble import GradientBoostingRegressor

from axle_studio.model import assumptions
from axle_studio.model.action import visible_day_ahead_prices
from axle_studio.model.availability import PRODUCT_WINDOWS
from axle_studio.model.forecast import run_forecast_from_assumptions
from axle_studio.model.sampling import day_ahead_publication_utc_ns
from axle_studio.ui.style import (
    LIGHT_INK,
    MUTED_INK,
    SERIES_COLOURS,
    apply_notebook_style,
    band_and_line,
)

NOTEBOOK_START = time.perf_counter()
# "notebook_connected" embeds only the figure data and loads plotly.js from a CDN, which keeps the
# committed notebook small. GitHub does not run that script, so every chart has a table under it.
pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 120)
{"scikit-learn": sklearn.__version__}

{'scikit-learn': '1.9.1'}

## Try it

The values below are "try it" settings for this notebook, not model assumptions: change them and
re-run. The run is small so the notebook stays within about a minute; the table under the next
cell reports the run time measured on this execution.

- `VEHICLE_COUNT`, `WORLD_COUNT`, `SEED`: the simulated fleet and how many weeks of "history" it
  gives. Forty weeks split 60/20/20 leaves eight test weeks.
- `SECOND_FLEET_VEHICLE_COUNT`: set it (for example to 600) to add a second run of a larger fleet
  with the same seed. Its rows join the history, so the `enrolled_evs` feature varies, and the
  fleet-size table in section 8 gains a second fleet size. Off by default because it roughly
  doubles the run time.
- `TARGET_DIRECTION`, `TARGET_DURATION_HOURS`: which deliverable to forecast (§10.2a: turn-down
  held for one hour by default).
- `GBR_PARAMS`: the gradient-boosting settings (contract §10.6a item 3's values).

In [2]:
START_LOCAL_DATE = date(2026, 1, 12)  # the London date the seven study nights start (a Monday)
VEHICLE_COUNT = 300  # large enough for a two-point fleet-size curve (100 and 300) in section 8
WORLD_COUNT = 40  # simulated weeks
SEED = 7
SECOND_FLEET_VEHICLE_COUNT = None  # try 600: a second, larger fleet with the same seed

TARGET_DIRECTION = "turn_down"
TARGET_DURATION_HOURS = 1.0
LEVELS = (0.1, 0.5, 0.9)
NOMINAL_COVERAGE = LEVELS[-1] - LEVELS[0]  # the P10-P90 band's nominal 80%
SPLIT_SHARES = {"train": 0.6, "calibration": 0.2, "test": 0.2}
GBR_PARAMS = {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.05, "subsample": 0.8}

fleet_sizes = [VEHICLE_COUNT] + ([SECOND_FLEET_VEHICLE_COUNT] if SECOND_FLEET_VEHICLE_COUNT else [])
runs = {}
run_seconds = {}
for fleet_size in fleet_sizes:
    started = time.perf_counter()
    runs[fleet_size] = run_forecast_from_assumptions(
        START_LOCAL_DATE,
        values={"vehicle_count": fleet_size, "evaluation_world_count": WORLD_COUNT, "seed": SEED},
    )
    run_seconds[fleet_size] = time.perf_counter() - started
base = runs[VEHICLE_COUNT]

# Read live from the assumption records, so the prose below never restates a stale default.
INTRADAY_DECISION_TIME = str(
    assumptions.AVAILABILITY["availability.intraday_decision_local_time"].value
)
pd.DataFrame(
    [
        {
            "vehicles": result.vehicle_count,
            "simulated weeks": result.world_count,
            "seed": SEED,
            "evidence_kind": result.evidence_kind,
            "intraday decision (London, record)": INTRADAY_DECISION_TIME,
            "run seconds": round(run_seconds[size], 1),
        }
        for size, result in runs.items()
    ]
)

,vehicles,simulated weeks,seed,evidence_kind,"intraday decision (London, record)",run seconds
0,300,40,7,illustrative_synthetic,17:00,45.6


## 1. The simulated history and its features

One history row is one (simulated week, night, half-hour). The target `y` is `deliverable_kw`
from `availability_world_slot`: the fleet's realised turn-down, in kW, that could be held for the
whole window starting in that half-hour (contract §10.2a).

**The information cut-off.** A day-ahead forecast of night `n` is made at 13:00 on the day before
(`τ_DA(n)`), one hour into night `n − 1` (nights run noon to noon). So a feature may use only what
is known then (contract §10.6a, review note B4):

| Feature | Meaning | Why it is known at `τ_DA(n)` |
| --- | --- | --- |
| `profile_order`, `sin_profile`, `cos_profile` | Position of the half-hour in the noon-to-noon night (0 = 12:00), and its cycle | Calendar |
| `weekday` | Weekday of the night's start date (0 = Monday) | Calendar |
| `holiday` | Holiday switch on that night (`world_nights.holiday`) | Calendar |
| `temperature_c`, `solar_clearness` | That evening's sampled weather (`solar_clearness` is dropped when the price generator has none, as the next cell reports) | **A perfect weather forecast**: these are the sampled values themselves, not a forecast with error |
| `enrolled_evs` | Fleet size (`vehicle_count`) | Known to the aggregator |
| `lag2_same_half_hour_kw` | Realised `y` two nights back at the same half-hour | Night `n − 2` ended at noon, before 13:00 |
| `lag2_evening_mean_kw` | Night `n − 2`'s mean `y` over the evening product window | Same |
| `price_rank_visible` (second top-down variant only) | Rank of the half-hour's day-ahead price within night `n` (1 = cheapest), from the prices visible at `τ_DA(n)` | The planner's and trader's own visibility rule (plan B4, `action.visible_day_ahead_prices`): prices published by 13:00 on the day before keep their value; the rest are the expected price shape |

**What the price feature can see.** Every price of delivery day `D` is published at 13:00 on
`D − 1` (`sampling.day_ahead_publication_utc_ns`). At `τ_DA(n)` that covers night `n`'s
half-hours up to midnight (delivery day `D_n`), but **not** those after midnight: their delivery
day is `D_n + 1`, published only at 13:00 on `D_n`. For those half-hours the rule substitutes the
**expected price shape**, the mean published price of the same half-hour over the most recent
published days, not tonight's price. So the rank is exact before midnight and a typical-night
guess after it, exactly what the smart charger ranks at that instant. One difference from the
planner: the result exposes day-ahead prices for the study week only (`forecast_prices`), not the
warm-up week, so the expected shape here averages over the published study days (one to five
nights' worth), where the planner's averages over seven days including the warm-up.

Night `n − 1` is still running at 13:00, so nothing from it is a feature; that is the same cut-off
the §4.1 baseline history uses. Nights 0 and 1 have no night `n − 2` and are dropped. No realised
quantity of night `n` itself is a feature. The lags are joined on (week, night, half-hour label),
not on a fixed 96-slot offset, so a clock-change night (46 or 50 half-hours) would still line up.

In [3]:
EVENING_START, EVENING_END = PRODUCT_WINDOWS["evening"]  # London wall-clock labels, read live
TARGET = "deliverable_kw"
INTRADAY_COLUMNS = ["intraday_p10_kw", "intraday_p50_kw", "intraday_p90_kw"]
FEATURES = [
    "profile_order",
    "sin_profile",
    "cos_profile",
    "weekday",
    "holiday",
    "temperature_c",
    "solar_clearness",
    "enrolled_evs",
    "lag2_same_half_hour_kw",
    "lag2_evening_mean_kw",
]
PRICE_FEATURE = "price_rank_visible"


def study_day_ahead_prices(result):
    """(world, study slot) day-ahead GBP/MWh and the slot starts (UTC), from ``forecast_prices``."""

    frame = result.forecast_prices.sort_values(["world_id", "slot_index"])
    starts = pd.DatetimeIndex(result.study_slots["interval_start_utc"])
    prices = frame["wholesale_forecast_gbp_per_mwh"].to_numpy(dtype=float)
    return prices.reshape(result.world_count, len(starts)), starts


def night_visible_rank(prices, starts, in_night):
    """Price rank (1 = cheapest) of each half-hour of one night, as visible at its tau_DA.

    ``prices`` is (world, study slot) GBP/MWh, ``in_night`` a bool mask of the night's slots.
    Returns the (world, night slot) ranks and the decision instant (int64 UTC ns).
    """

    # tau_DA(n) is when the night's first (12:00) half-hour is published: 13:00 on the day before.
    decision_ns = int(day_ahead_publication_utc_ns(starts[in_night][:1])[0])
    visible = visible_day_ahead_prices(prices, starts, decision_ns)[:, in_night]
    return rankdata(visible, axis=1, method="average"), decision_ns


def visible_price_rank(result):
    """Long frame (world_id, slot_index, price_rank_visible, price_published_by_decision)."""

    prices, starts = study_day_ahead_prices(result)
    nights = result.study_slots["night_index"].to_numpy()
    publication = day_ahead_publication_utc_ns(starts)
    rank = np.full(prices.shape, np.nan)
    published = np.zeros(len(starts), dtype=bool)
    for night in np.unique(nights[nights >= 2]):
        in_night = nights == night
        rank[:, in_night], decision_ns = night_visible_rank(prices, starts, in_night)
        published[in_night] = publication[in_night] <= decision_ns
    return pd.DataFrame(
        {
            "world_id": np.repeat(np.arange(result.world_count), len(starts)),
            "slot_index": np.tile(result.study_slots["slot_index"].to_numpy(), result.world_count),
            PRICE_FEATURE: rank.ravel(),
            "price_published_by_decision": np.tile(published, result.world_count),
        }
    )


def history_rows(result):
    """(week, night, half-hour) rows for one run: target, day-ahead features, intraday band.

    ``deliverable_kw`` and the intraday columns are kW for ``TARGET_DIRECTION`` held for
    ``TARGET_DURATION_HOURS``. Nights 0 and 1 are dropped (no night n - 2), as are windows running
    past the study end (NaN target).
    """

    slots = result.availability_world_slot
    rows = slots.loc[
        (slots["direction"] == TARGET_DIRECTION)
        & (slots["duration_hours"] == TARGET_DURATION_HOURS),
        ["world_id", "slot_index", "night_index", TARGET, *INTRADAY_COLUMNS],
    ].merge(
        result.study_slots[
            ["slot_index", "local_half_hour", "local_time_label", "interval_start_london"]
        ],
        on="slot_index",
    )
    # Noon-to-noon position, the order every overnight profile in the app uses (12:00 is 0).
    rows["profile_order"] = (rows["local_half_hour"] - 24) % 48
    angle = 2 * np.pi * rows["profile_order"] / 48
    rows["sin_profile"] = np.sin(angle)
    rows["cos_profile"] = np.cos(angle)

    nights = result.world_nights[
        [
            "world_id",
            "night_index",
            "night_start_local_date",
            "holiday",
            "temperature_c",
            "solar_clearness",
        ]
    ].copy()
    nights["weekday"] = pd.to_datetime(nights["night_start_local_date"]).dt.weekday
    nights["holiday"] = nights["holiday"].astype(int)
    rows = rows.merge(nights, on=["world_id", "night_index"])
    rows["enrolled_evs"] = result.vehicle_count

    # Lags come from night n - 2 only: complete at 13:00 on the day before night n (B4).
    same_half_hour = (
        rows.groupby(["world_id", "night_index", "local_time_label"])[TARGET]
        .mean()  # mean only matters on a repeated autumn half-hour label
        .rename("lag2_same_half_hour_kw")
        .reset_index()
    )
    in_evening = (rows["local_time_label"] >= EVENING_START) & (
        rows["local_time_label"] < EVENING_END
    )
    evening_mean = (
        rows.loc[in_evening]
        .groupby(["world_id", "night_index"])[TARGET]
        .mean()
        .rename("lag2_evening_mean_kw")
        .reset_index()
    )
    same_half_hour["night_index"] += 2
    evening_mean["night_index"] += 2
    rows = rows.merge(
        same_half_hour, on=["world_id", "night_index", "local_time_label"], how="left"
    )
    rows = rows.merge(evening_mean, on=["world_id", "night_index"], how="left")
    rows = rows.merge(visible_price_rank(result), on=["world_id", "slot_index"], how="left")
    return rows.loc[(rows["night_index"] >= 2) & rows[TARGET].notna()].reset_index(drop=True)


history = pd.concat([history_rows(result) for result in runs.values()], ignore_index=True)
if history["solar_clearness"].isna().all():
    # This price generator has no daily clearness, so world_nights carries NaN (§10.1f allows
    # it). An all-NaN column carries no information; drop it rather than impute a number.
    FEATURES.remove("solar_clearness")
ALL_FEATURES = [*FEATURES, PRICE_FEATURE]
print(f"{len(history):,} history rows; features: {', '.join(ALL_FEATURES)}")
history[["world_id", "night_index", "local_time_label", TARGET, *ALL_FEATURES]].head(8).round(2)

9,560 history rows; features: profile_order, sin_profile, cos_profile, weekday, holiday, temperature_c, enrolled_evs, lag2_same_half_hour_kw, lag2_evening_mean_kw, price_rank_visible


,world_id,night_index,local_time_label,deliverable_kw,profile_order,sin_profile,cos_profile,weekday,holiday,temperature_c,enrolled_evs,lag2_same_half_hour_kw,lag2_evening_mean_kw,price_rank_visible
0,0,2,12:00,0.0,0,0.00,1.00,2,0,11.42,300,0.0,0.0,31.0
1,0,2,12:30,0.0,1,0.13,0.99,2,0,11.42,300,0.0,0.0,26.0
2,0,2,13:00,0.0,2,0.26,0.97,2,0,11.42,300,0.0,0.0,19.0
3,0,2,13:30,0.0,3,0.38,0.92,2,0,11.42,300,0.0,0.0,7.0
4,0,2,14:00,0.0,4,0.50,0.87,2,0,11.42,300,0.0,0.0,22.0
5,0,2,14:30,0.0,5,0.61,0.79,2,0,11.42,300,0.0,0.0,27.0
6,0,2,15:00,0.0,6,0.71,0.71,2,0,11.42,300,0.0,0.0,34.0
7,0,2,15:30,0.0,7,0.79,0.61,2,0,11.42,300,0.0,0.0,46.0


## 2. Leave-weeks-out split

The split is by simulated week, not by row. The half-hours of one week share its weather, its
plug-in week factor, its maker outages and its price path, so rows of one week are strongly
dependent; a row-level split would let the model learn a test week from its neighbours (leakage)
and flatter every score. Sixty per cent of the weeks train, twenty per cent calibrate the
conformal correction, twenty per cent are held out for the comparison. With a second fleet, the
same week id falls in the same part for both fleets, because both runs share the seed and so the
week's exogenous draws.

**Which half-hours are used.** Only the half-hours of the evening and overnight product windows
(`PRODUCT_WINDOWS`, read live: 17:00–21:00 and 21:00–06:00 at the time of writing), the hours a
turn-down product is sold for. Daytime half-hours with no home charging deliver 0 kW in almost
every week; keeping them would pad every method's coverage with free hits and pile the conformal
scores onto an exact zero. The rule is calendar only, so it cannot favour any method.

In [4]:
split_rng = np.random.default_rng(SEED)
shuffled_worlds = split_rng.permutation(np.sort(history["world_id"].unique()))
n_train = round(SPLIT_SHARES["train"] * len(shuffled_worlds))
n_calibration = round(SPLIT_SHARES["calibration"] * len(shuffled_worlds))
split_worlds = {
    "train": shuffled_worlds[:n_train],
    "calibration": shuffled_worlds[n_train : n_train + n_calibration],
    "test": shuffled_worlds[n_train + n_calibration :],
}
history["split"] = history["world_id"].map(
    {world: part for part, worlds in split_worlds.items() for world in worlds}
)


def in_window(labels, window):
    """True where a London "HH:MM" label lies in a ``PRODUCT_WINDOWS`` window."""

    start, end = PRODUCT_WINDOWS[window]
    if start < end:
        return (labels >= start) & (labels < end)
    return (labels >= start) | (labels < end)  # wraps midnight


SCORED_WINDOWS = ("evening", "overnight")
in_scored_window = np.logical_or.reduce(
    [in_window(history["local_time_label"], window) for window in SCORED_WINDOWS]
)
history = history.loc[in_scored_window].reset_index(drop=True)
print(
    "Scored half-hours whose price rank uses the published price (the rest use the expected "
    f"shape): {history['price_published_by_decision'].mean():.0%}"
)

pd.DataFrame(
    {
        "weeks": {part: len(worlds) for part, worlds in split_worlds.items()},
        "rows": history["split"].value_counts(),
        "mean y (kW)": history.groupby("split")[TARGET].mean().round(1),
    }
).loc[list(split_worlds)]

Scored half-hours whose price rank uses the published price (the rest use the expected shape): 54%


,weeks,rows,mean y (kW)
train,24,3120,106.4
calibration,8,1040,102.4
test,8,1040,84.1


## 3. The top-down model: gradient-boosted quantiles

For each level `α` in 0.1, 0.5 and 0.9 a separate `GradientBoostingRegressor(loss="quantile",
alpha=α)` is fitted on the training weeks. The quantile (pinball) loss

$$
\rho_\alpha(y, q) = \max\{\alpha (y - q),\ (\alpha - 1)(y - q)\}
$$

is minimised by the true `α`-quantile, so each model learns a conditional quantile directly, with
no distributional assumption. Two variants are fitted with the same settings: **without price**
(the section 1 features) and **with price** (the same plus `price_rank_visible`). Two small repairs, both standard: the three separately fitted
quantiles can cross on some rows, so each row's three values are sorted; and a deliverable cannot
be negative, so predictions are floored at 0 kW.

In [5]:
def fit_quantile_models(rows, features):
    """One gradient-boosted quantile model per level in ``LEVELS``, fitted on ``rows``."""

    return {
        level: GradientBoostingRegressor(
            loss="quantile", alpha=level, random_state=SEED, **GBR_PARAMS
        ).fit(rows[features], rows[TARGET])
        for level in LEVELS
    }


def predict_quantiles(models, rows, features):
    """(row, level) kW: sorted across levels (no crossing) and floored at 0."""

    raw = np.column_stack([models[level].predict(rows[features]) for level in LEVELS])
    return np.maximum(np.sort(raw, axis=1), 0.0)


VARIANTS = {"top_down": FEATURES, "top_down_price": ALL_FEATURES}
QUANTILE_NAMES = [f"P{round(level * 100)}" for level in LEVELS]
train = history.loc[history["split"] == "train"]
importance = {}
started = time.perf_counter()
for variant, features in VARIANTS.items():
    models = fit_quantile_models(train, features)
    history[[f"{variant}_p{round(level * 100)}" for level in LEVELS]] = predict_quantiles(
        models, history, features
    )
    importance["with price" if variant.endswith("price") else "without price"] = pd.DataFrame(
        {name: models[level].feature_importances_ for name, level in zip(QUANTILE_NAMES, LEVELS)},
        index=features,
    )
fit_seconds = time.perf_counter() - started
importance = pd.concat(importance, axis=1).reindex(ALL_FEATURES)  # NaN: feature not in variant

train_zero_share = float((train[TARGET] == 0).mean())
print(f"Fitted {len(VARIANTS) * len(LEVELS)} models in {fit_seconds:.1f} s")
print(f"Training rows at exactly 0 kW: {train_zero_share:.0%}")
for variant, name in importance.columns:
    column = importance[(variant, name)]
    if column.sum() == 0:
        print(f"{variant}, {name}: no split improved the loss, so the model predicts one constant")
    else:
        print(f"{variant}, {name}: leans most on {column.idxmax()}")
test_rows = history["split"] == "test"
for variant in VARIANTS:
    print(
        f"{variant} after sort and floor, test rows: P10 exactly 0 kW on "
        f"{(history.loc[test_rows, f'{variant}_p10'] == 0).mean():.0%}; "
        f"{history.loc[test_rows, f'{variant}_p50'].nunique()} distinct stored P50 values"
    )
importance.round(3)

Fitted 6 models in 3.1 s
Training rows at exactly 0 kW: 53%
without price, P10: leans most on temperature_c
without price, P50: no split improved the loss, so the model predicts one constant
without price, P90: leans most on temperature_c
with price, P10: leans most on price_rank_visible
with price, P50: no split improved the loss, so the model predicts one constant
with price, P90: leans most on price_rank_visible
top_down after sort and floor, test rows: P10 exactly 0 kW on 100%; 3 distinct stored P50 values
top_down_price after sort and floor, test rows: P10 exactly 0 kW on 100%; 49 distinct stored P50 values


without price             with price            
                                 P10  P50    P90        P10  P50    P90
profile_order                  0.140  0.0  0.098      0.073  0.0  0.063
sin_profile                    0.105  0.0  0.063      0.099  0.0  0.062
cos_profile                    0.099  0.0  0.220      0.039  0.0  0.235
weekday                        0.038  0.0  0.110      0.076  0.0  0.054
holiday                        0.000  0.0  0.000      0.000  0.0  0.000
temperature_c                  0.395  0.0  0.237      0.191  0.0  0.137
enrolled_evs                   0.000  0.0  0.000      0.000  0.0  0.000
lag2_same_half_hour_kw         0.134  0.0  0.182      0.048  0.0  0.080
lag2_evening_mean_kw           0.088  0.0  0.090      0.073  0.0  0.097
price_rank_visible               NaN  NaN    NaN      0.401  0.0  0.271

The importance table says what each quantile model leans on (impurity importance, shares summing
to 1 per level). Read it with the zero share printed above: at a small fleet, 1-h turn-down is
exactly 0 kW in well over half of the window half-hours (section 5b shows why), so any quantile at
or below that share is 0 kW almost everywhere, and a model for it has nothing to learn. The
information sits in the upper quantiles. Compare the two variants to see how much of the model's
attention the visible price rank takes once it is offered. The row-wise sort and the 0 kW floor
change what is stored: as printed above, the stored P10 is exactly 0 kW on every test row, while
the stored P50 is not constant even though the raw P50 model is, because sorting moves a raw P10
or P90 into the middle position wherever it crosses the constant.

## 4. The conformal correction

A quantile model fitted on a few dozen weeks is often over-confident: its P10–P90 band can cover
fewer than 80% of new rows. Split conformal prediction checks and repairs the width with one
number learned on the calibration weeks, which the model never saw (contract §10.6a item 4):

$$
E_i = \max(\hat q_{0.1}(x_i) - y_i,\ y_i - \hat q_{0.9}(x_i)), \qquad
Q = \text{the } \lceil (n + 1)(1 - 0.2) \rceil / n \text{ empirical quantile of } E
$$

and the corrected band is `[q̂_0.1 − Q, q̂_0.9 + Q]`, floored at 0 kW. `E_i` is positive when
`y_i` falls outside the raw band (by how far) and negative inside it, so `Q > 0` widens an
over-confident band and `Q < 0` would narrow an over-cautious one. On exchangeable rows the
corrected band has at least 80% coverage. Rows within one week are not exchangeable (they share
the week's factors), and only a handful of weeks calibrate, so here the guarantee is approximate;
section 9 asserts a tolerant band rather than exactly 80% (contract review note O7). The P50 is not
touched: CQR corrects the band, not the median.

In [6]:
def conformal_margin(low_kw, high_kw, y_kw, coverage):
    """Split-conformal margin Q (kW) for a [low, high] band at the given nominal coverage.

    Romano, Patterson and Candès (2019): the finite-sample-corrected quantile of the
    conformity scores E = max(low - y, y - high) on held-out calibration rows.
    """

    scores = np.maximum(low_kw - y_kw, y_kw - high_kw)
    n = len(scores)
    level = min(1.0, np.ceil((n + 1) * coverage) / n)
    return float(np.quantile(scores, level, method="higher"))


calibration = history["split"] == "calibration"
margins_kw, raw_calibration_coverage = {}, {}
for variant in VARIANTS:
    corrected = variant.replace("top_down", "conformal")
    low, mid, high = (history[f"{variant}_p{round(level * 100)}"] for level in LEVELS)
    margins_kw[variant] = conformal_margin(
        low[calibration].to_numpy(),
        high[calibration].to_numpy(),
        history.loc[calibration, TARGET].to_numpy(),
        NOMINAL_COVERAGE,
    )
    history[f"{corrected}_p10"] = np.maximum(low - margins_kw[variant], 0.0)
    history[f"{corrected}_p50"] = mid
    history[f"{corrected}_p90"] = high + margins_kw[variant]
    raw_calibration_coverage[variant] = float(
        history.loc[calibration, TARGET].between(low[calibration], high[calibration]).mean()
    )

pd.DataFrame(
    {
        "calibration rows": int(calibration.sum()),
        "raw P10-P90 coverage on calibration weeks": raw_calibration_coverage,
        "conformal margin Q (kW)": margins_kw,
    }
).round(3)

,calibration rows,raw P10-P90 coverage on calibration weeks,conformal margin Q (kW)
top_down,1040,0.896,0.0
top_down_price,1040,0.915,0.0


In [7]:
# Say what the correction did in this run, rather than assume the textbook case.
for variant, margin_kw in margins_kw.items():
    if margin_kw > 0:
        print(f"{variant}: Q = {margin_kw:.2f} kW > 0, the raw band was over-confident; widened.")
    elif margin_kw == 0:
        print(
            f"{variant}: Q = 0. The raw band already covered "
            f"{raw_calibration_coverage[variant]:.0%} of calibration rows; coverage of at least "
            "80% on its own gives Q <= 0. Every 0 kW row inside a band starting at 0 kW scores "
            "E = 0 exactly, so the 80% quantile of E lands on that mass and Q is exactly 0 "
            "rather than negative: the band is left unchanged."
        )
    else:
        print(f"{variant}: Q = {margin_kw:.2f} kW < 0, the raw band was too cautious; narrowed.")

top_down: Q = 0. The raw band already covered 90% of calibration rows; coverage of at least 80% on its own gives Q <= 0. Every 0 kW row inside a band starting at 0 kW scores E = 0 exactly, so the 80% quantile of E lands on that mass and Q is exactly 0 rather than negative: the band is left unchanged.
top_down_price: Q = 0. The raw band already covered 92% of calibration rows; coverage of at least 80% on its own gives Q <= 0. Every 0 kW row inside a band starting at 0 kW scores E = 0 exactly, so the 80% quantile of E lands on that mass and Q is exactly 0 rather than negative: the band is left unchanged.


## 5. Held-out comparison: coverage, pinball loss, sharpness

Six forecasts of the same test-week rows:

- **(a) Bottom-up day-ahead (training weeks).** The §10.3 rule with the test weeks left out: the
  P10, P50 and P90 of `deliverable_kw` across the *training* weeks at the same half-hour of the
  study (linear `numpy.quantile`, as `availability_bands` uses). It knows nothing about the test
  week, not even its weather.
- **(b) Bottom-up intraday.** The run's own `intraday_p10/p50/p90_kw` (§10.2d): the whole night
  re-forecast at the intraday decision time from the EVs already plugged in plus the other weeks'
  late arrivals. It has later information (the driveway at the decision time), so it is **not a
  like-for-like competitor** of the day-ahead forecasts; it exists only from the decision
  half-hour onwards and is scored on the rows where it exists (all of them while the decision time
  is the evening window's start).
- **(c) Top-down raw** and **(d) top-down + conformal**, without the price feature, and
  **(c′)** and **(d′)**, the same with `price_rank_visible`: the before and after of adding it.

**How to read the scores.** Coverage is the share of test rows whose realised value lies inside
the P10–P90 band; a calibrated band covers about 80%, i.e. about one row in five falls outside.
Pinball loss (kW, lower is better) scores each quantile against the outcome and rewards being both
calibrated and sharp; at P50 it is half the absolute error. Sharpness is the mean P10–P90 width in
kW: narrower is better *only* at the same coverage, so the two are always read together.

In [8]:
def training_week_band(rows):
    """(a) Bottom-up day-ahead P10/P50/P90 (kW) per (fleet, study slot) over the training weeks."""

    train = rows.loc[rows["split"] == "train"]
    return (
        train.groupby(["enrolled_evs", "slot_index"])[TARGET]
        .quantile(list(LEVELS))  # pandas' default "linear", the same rule as numpy.quantile
        .unstack()
        .set_axis(["bottom_up_p10", "bottom_up_p50", "bottom_up_p90"], axis=1)
    )


history = history.join(training_week_band(history), on=["enrolled_evs", "slot_index"])
history = history.rename(
    columns=dict(zip(INTRADAY_COLUMNS, ["intraday_p10", "intraday_p50", "intraday_p90"]))
)

METHODS = {
    "bottom_up": "(a) Bottom-up day-ahead, training weeks",
    "intraday": f"(b) Bottom-up intraday, from {INTRADAY_DECISION_TIME}",
    "top_down": "(c) Top-down GBR, raw, no price",
    "conformal": "(d) Top-down GBR + conformal, no price",
    "top_down_price": "(c') Top-down GBR, raw, visible price rank",
    "conformal_price": "(d') Top-down GBR + conformal, visible price rank",
}


def pinball_kw(y, q, level):
    """Mean pinball loss (kW) of quantile forecast ``q`` at ``level`` for outcomes ``y``."""

    difference = y - q
    return float(np.mean(np.maximum(level * difference, (level - 1) * difference)))


def score(rows, method):
    """Coverage, per-level pinball and sharpness of one method's P10/P50/P90 on ``rows``."""

    y = rows[TARGET].to_numpy()
    low, mid, high = (rows[f"{method}_p{round(level * 100)}"].to_numpy() for level in LEVELS)
    return {
        "rows": len(rows),
        "coverage P10-P90": float(np.mean((y >= low) & (y <= high))),
        "below P10": float(np.mean(y < low)),
        "above P90": float(np.mean(y > high)),
        "pinball P10 (kW)": pinball_kw(y, low, LEVELS[0]),
        "pinball P50 (kW)": pinball_kw(y, mid, LEVELS[1]),
        "pinball P90 (kW)": pinball_kw(y, high, LEVELS[2]),
        "sharpness P10-P90 (kW)": float(np.mean(high - low)),
    }


test = history.loc[history["split"] == "test"]
test_intraday = test.loc[test["intraday_p50"].notna()]
comparison = pd.DataFrame(
    [
        {"method": METHODS[m], **score(test_intraday if m == "intraday" else test, m)}
        for m in METHODS
    ]
).set_index("method")
# Mean pinball over the three levels: one proper score per method, the benchmark test below.
comparison["mean pinball (kW)"] = comparison[
    ["pinball P10 (kW)", "pinball P50 (kW)", "pinball P90 (kW)"]
].mean(axis=1)
comparison.round(3)

,rows,coverage P10-P90,below P10,above P90,pinball P10 (kW),pinball P50 (kW),pinball P90 (kW),sharpness P10-P90 (kW),mean pinball (kW)
method,,,,,,,,,
"(a) Bottom-up day-ahead, training weeks",1040,0.904,0.001,0.095,8.41,44.419,41.969,320.485,31.599
"(b) Bottom-up intraday, from 17:00",1040,0.908,0.002,0.090,7.64,39.455,36.254,304.609,27.783
"(c) Top-down GBR, raw, no price",1040,0.913,0.000,0.087,8.41,42.047,41.730,323.858,30.729
"(d) Top-down GBR + conformal, no price",1040,0.913,0.000,0.087,8.41,42.047,41.730,323.858,30.729
"(c') Top-down GBR, raw, visible price rank",1040,0.922,0.000,0.078,8.41,42.026,38.456,331.287,29.631
"(d') Top-down GBR + conformal, visible price rank",1040,0.922,0.000,0.078,8.41,42.026,38.456,331.287,29.631


Read the table row pair by row pair, not as a league: a method with lower pinball loss *and*
coverage near 80% is better; a narrower band with coverage well under 80% is over-confident, not
better. The printed findings below are computed from this run, so they change when the "try it"
values do. The last lines ask whether the top-down bands really beat (a): not from the pooled
table, but week by week, with a standard error across the eight test weeks.

In [9]:
for method in METHODS:
    row = comparison.loc[METHODS[method]]
    outside = 1 - row["coverage P10-P90"]
    print(
        f"{METHODS[method]}: {row['coverage P10-P90']:.0%} covered "
        f"(about {outside * 10:.1f} rows in 10 outside), "
        f"mean width {row['sharpness P10-P90 (kW)']:.1f} kW, "
        f"P50 pinball {row['pinball P50 (kW)']:.2f} kW"
    )


# Is the bottom-up day-ahead band (a) still the benchmark? A difference of two pooled scores
# hides how much it moves from week to week, so compare week by week: each test week's mean
# pinball (over its rows and the three levels) for (d) and (d'), minus the same week's for (a).
# Paired by week, so the week's own weather, prices and outages cancel.
def row_mean_pinball(rows, method):
    y = rows[TARGET].to_numpy()
    losses = []
    for level in LEVELS:
        difference = y - rows[f"{method}_p{round(level * 100)}"].to_numpy()
        losses.append(np.maximum(level * difference, (level - 1) * difference))
    return np.mean(losses, axis=0)  # per row, the mean over the three levels


weekly = pd.DataFrame(
    {
        method: pd.Series(row_mean_pinball(test, method), index=test.index)
        .groupby(test["world_id"])
        .mean()
        for method in ("bottom_up", "conformal", "conformal_price")
    }
)
weekly_difference = pd.DataFrame(
    {
        "(d) - (a) (kW)": weekly["conformal"] - weekly["bottom_up"],
        "(d') - (a) (kW)": weekly["conformal_price"] - weekly["bottom_up"],
        "test week mean y (kW)": test.groupby("world_id")[TARGET].mean(),
    }
)
for column in ("(d) - (a) (kW)", "(d') - (a) (kW)"):
    difference = weekly_difference[column]
    mean, se = difference.mean(), difference.std(ddof=1) / np.sqrt(len(difference))
    # Two standard errors as the plain noise yardstick: eight weeks support no finer claim.
    reading = "within week-to-week noise" if abs(mean) < 2 * se else "beyond week-to-week noise"
    print(
        f"{column[:-5]}: lower in {int((difference < 0).sum())} of {len(difference)} weeks, "
        f"mean difference {mean:.2f} ± {se:.2f} kW (SE), {reading}"
    )
print(
    f"Mean y: test weeks {test[TARGET].mean():.1f} kW, training weeks "
    f"{history.loc[history['split'] == 'train', TARGET].mean():.1f} kW: the test weeks differ "
    "from the training level, which moves every method's scores."
)
weekly_difference.round(2)

(a) Bottom-up day-ahead, training weeks: 90% covered (about 1.0 rows in 10 outside), mean width 320.5 kW, P50 pinball 44.42 kW
(b) Bottom-up intraday, from 17:00: 91% covered (about 0.9 rows in 10 outside), mean width 304.6 kW, P50 pinball 39.45 kW
(c) Top-down GBR, raw, no price: 91% covered (about 0.9 rows in 10 outside), mean width 323.9 kW, P50 pinball 42.05 kW
(d) Top-down GBR + conformal, no price: 91% covered (about 0.9 rows in 10 outside), mean width 323.9 kW, P50 pinball 42.05 kW
(c') Top-down GBR, raw, visible price rank: 92% covered (about 0.8 rows in 10 outside), mean width 331.3 kW, P50 pinball 42.03 kW
(d') Top-down GBR + conformal, visible price rank: 92% covered (about 0.8 rows in 10 outside), mean width 331.3 kW, P50 pinball 42.03 kW
(d) - (a): lower in 5 of 8 weeks, mean difference -0.87 ± 0.68 kW (SE), within week-to-week noise
(d') - (a): lower in 8 of 8 weeks, mean difference -1.97 ± 0.59 kW (SE), beyond week-to-week noise
Mean y: test weeks 84.1 kW, training weeks

,(d) - (a) (kW),(d') - (a) (kW),test week mean y (kW)
world_id,,,
2,2.18,-0.36,110.82
5,-2.80,-3.80,64.21
11,0.80,-0.03,104.16
21,-0.66,-0.92,76.40
23,-2.38,-4.31,106.84
25,-3.02,-3.46,13.79
31,-1.67,-1.71,105.19
34,0.58,-1.15,91.39


### 5b. What drives the misses: tonight's prices

Why is the deliverable so often exactly 0 kW, and why is it hard to forecast a day ahead? The
smart plans put charging in the cheapest half-hours of each night (the planner's own objective),
and the cheapest half-hours move from week to week with the simulated prices. A half-hour that is
cheap tonight has many EVs charging and so something to turn down; one that is dear has none. The
table ranks each held-out night's scored half-hours by that week's day-ahead price (1 = cheapest)
and shows the realised deliverable. The bottom-up band (a) pools all weeks and so cannot see
tonight's prices; variant (d′) sees them through `price_rank_visible`, exact before midnight and
the expected shape after it (section 1). The ranking in this table uses the week's actual
day-ahead prices for every half-hour, after the fact, to explain the misses; it is not a feature.

In [10]:
prices = base.deviation_world_slot[["world_id", "slot_index", "day_ahead_gbp_per_mwh"]]
ranked = test.loc[test["enrolled_evs"] == VEHICLE_COUNT].merge(
    prices, on=["world_id", "slot_index"]
)
# 1 = the night's cheapest scored half-hour; this week's prices, used only to explain the misses.
ranked["price_rank"] = ranked.groupby(["world_id", "night_index"])["day_ahead_gbp_per_mwh"].rank(
    method="first"
)
ranked["price rank in the night"] = pd.cut(
    ranked["price_rank"], [0, 4, 8, 12, 16, np.inf], labels=["1-4", "5-8", "9-12", "13-16", "17+"]
)


def rank_summary(rows):
    y = rows[TARGET]
    return pd.Series(
        {
            "half-hours": len(rows),
            "realised mean (kW)": y.mean(),
            "share at 0 kW": (y == 0).mean(),
            "(a) outside band": (~y.between(rows["bottom_up_p10"], rows["bottom_up_p90"])).mean(),
            "(d) outside band": (~y.between(rows["conformal_p10"], rows["conformal_p90"])).mean(),
            "(d') outside band": (
                ~y.between(rows["conformal_price_p10"], rows["conformal_price_p90"])
            ).mean(),
        }
    )


price_table = ranked.groupby("price rank in the night", observed=True).apply(
    rank_summary, include_groups=False
)
price_table.round(3)

,half-hours,realised mean (kW),share at 0 kW,(a) outside band,(d) outside band,(d') outside band
price rank in the night,,,,,,
1-4,160.0,327.628,0.162,0.419,0.400,0.300
5-8,160.0,144.664,0.319,0.150,0.106,0.106
9-12,160.0,55.501,0.375,0.038,0.038,0.050
13-16,160.0,13.576,0.612,0.012,0.012,0.019
17+,400.0,2.113,0.912,0.002,0.002,0.012


## 6. One test week, drawn

The first held-out week: the realised deliverable (one simulated week, a single draw), the
bottom-up day-ahead band from the training weeks, and the top-down conformal band with the
visible price rank (d′). Each band is
P10–P90 with its P50 as the line; in a calibrated band about one half-hour in five falls outside.
The bottom-up band follows the time of night, the same for every week; the top-down band also
moves with this week's weather, what the fleet did two nights earlier and where the visible
prices put the cheap half-hours.

In [11]:
week = int(split_worlds["test"][0])
week_rows = test.loc[
    (test["world_id"] == week) & (test["enrolled_evs"] == VEHICLE_COUNT)
].sort_values("slot_index")
# Every study slot of nights 2-6 on the x axis, so the dropped afternoon half-hours leave a gap
# in the lines instead of joining one night to the next.
week_plot = base.study_slots.loc[
    base.study_slots["night_index"] >= 2, ["slot_index", "interval_start_london"]
].merge(week_rows.drop(columns="interval_start_london"), on="slot_index", how="left")

week_figure = go.Figure()
band_and_line(
    week_figure,
    week_plot["interval_start_london"],
    week_plot["bottom_up_p10"],
    week_plot["bottom_up_p50"],
    week_plot["bottom_up_p90"],
    name="Bottom-up day-ahead, P50 in P10-P90 across training weeks",
    colour=SERIES_COLOURS["selected"],
)
band_and_line(
    week_figure,
    week_plot["interval_start_london"],
    week_plot["conformal_price_p10"],
    week_plot["conformal_price_p50"],
    week_plot["conformal_price_p90"],
    name="Top-down + conformal with visible price rank, P50 in P10-P90",
    colour=SERIES_COLOURS["difference"],
    dash="dot",
)
week_figure.add_trace(
    go.Scatter(
        x=week_plot["interval_start_london"],
        y=week_plot[TARGET],
        mode="lines",
        line={"color": LIGHT_INK, "width": 1.5},
        name="Realised (this simulated week)",
    )
)
week_figure.update_layout(
    title=f"Held-out simulated week {week}: realised 1-h turn-down against both bands",
    xaxis_title="London local time",
    yaxis_title="Deliverable turn-down (kW)",
    yaxis_rangemode="tozero",
)
apply_notebook_style(week_figure, height=420)
week_figure.show()

In [12]:
def night_summary(rows):
    """Per night of the drawn week: realised mean and, per band, rows outside and mean width."""

    out = {"half-hours": len(rows), "realised mean (kW)": rows[TARGET].mean()}
    for method in ("bottom_up", "conformal_price"):
        low, high = rows[f"{method}_p10"], rows[f"{method}_p90"]
        out[f"{method}: outside band"] = int((~rows[TARGET].between(low, high)).sum())
        out[f"{method}: mean width (kW)"] = (high - low).mean()
    return pd.Series(out)


week_rows.groupby("night_index").apply(night_summary, include_groups=False).round(1)

,half-hours,realised mean (kW),bottom_up: outside band,bottom_up: mean width (kW),conformal_price: outside band,conformal_price: mean width (kW)
night_index,,,,,,
2,26.0,50.5,2.0,283.6,2.0,332.8
3,26.0,112.3,2.0,272.0,2.0,271.2
4,26.0,153.1,7.0,384.4,4.0,255.0
5,26.0,47.5,1.0,354.0,1.0,370.2
6,26.0,170.8,4.0,308.4,3.0,305.5


## 7. Reliability

A reliability diagram plots, for each level `α`, the share of test rows at or below the forecast
`α`-quantile against `α` itself. A calibrated forecast sits on the diagonal. Above the diagonal at
P10 means the lower edge is too low (too cautious); below the diagonal at P90 means the upper edge
is too low (over-confident). With a point mass at 0 kW (an EV-less half-hour), a calibrated
forecast can sit above the diagonal at low levels because a 0 kW outcome ties a 0 kW quantile;
the backtest frames report `coverage_strict` for that reason (§10.3).

In [13]:
def reliability(rows, method):
    y = rows[TARGET].to_numpy()
    return [
        float(np.mean(y <= rows[f"{method}_p{round(level * 100)}"].to_numpy())) for level in LEVELS
    ]


reliability_table = pd.DataFrame(
    {
        METHODS[m]: reliability(test_intraday if m == "intraday" else test, m)
        for m in ("bottom_up", "intraday", "conformal", "conformal_price")
    },
    index=pd.Index(LEVELS, name="nominal level"),
)

reliability_figure = go.Figure()
reliability_figure.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        line={"color": MUTED_INK, "dash": "dash", "width": 1},
        name="Perfect calibration",
    )
)
method_colours = {
    "bottom_up": SERIES_COLOURS["selected"],
    "intraday": SERIES_COLOURS["synthetic"],
    "conformal": SERIES_COLOURS["normal"],
    "conformal_price": SERIES_COLOURS["difference"],
}
for method, colour in method_colours.items():
    reliability_figure.add_trace(
        go.Scatter(
            x=list(LEVELS),
            y=reliability_table[METHODS[method]],
            mode="lines+markers",
            line={"color": colour},
            name=METHODS[method],
        )
    )
reliability_figure.update_layout(
    title="Held-out weeks: observed share at or below each forecast quantile",
    xaxis_title="Nominal level",
    yaxis_title="Observed share of test rows",
    xaxis_range=[0, 1],
    yaxis_range=[0, 1],
)
apply_notebook_style(reliability_figure, height=420)
reliability_figure.show()
reliability_table.round(3)

,"(a) Bottom-up day-ahead, training weeks","(b) Bottom-up intraday, from 17:00","(d) Top-down GBR + conformal, no price","(d') Top-down GBR + conformal, visible price rank"
nominal level,,,,
0.1,0.577,0.577,0.577,0.577
0.5,0.643,0.631,0.578,0.578
0.9,0.905,0.910,0.913,0.922


The intraday line (b) is scored on the half-hours from the intraday decision onwards only, so its
points may not be on the same rows as the others (while the decision time is the evening
window's start they are). The raw variants (c) and (c′) are in the section 5 table.

## 8. Firm share against fleet size

`firm_share = P10 ÷ P50` of the fleet deliverable across weeks (§10.2c): the share of the typical
deliverable that nine weeks in ten can count on. `firm_share_by_fleet_size` computes it for
sub-fleets of the run (the first `n` EVs, each of 100, 300, 1,000, 3,000 up to the fleet size),
at no extra simulation cost. Independent EVs would push the firm share towards 1 as the fleet
grows; the shared day and week plug-in factors, weather and maker outages keep it below 1. At the
default 300 EVs the frame has two fleet sizes, 100 and 300; set `SECOND_FLEET_VEHICLE_COUNT` in
"Try it" for a larger run (a larger `N` is a new run, contract §10.6b).

In [14]:
fleet_curve = pd.concat(
    [
        result.firm_share_by_fleet_size.assign(run_vehicles=result.vehicle_count)
        for result in runs.values()
    ],
    ignore_index=True,
)
fleet_curve = fleet_curve.loc[
    (fleet_curve["direction"] == TARGET_DIRECTION)
    & (fleet_curve["duration_hours"] == TARGET_DURATION_HOURS)
]
# The largest run's own curve: every sub-fleet shares that run's weeks (paired), so its points
# differ only in fleet size.
largest_curve = fleet_curve.loc[fleet_curve["run_vehicles"] == max(runs)].sort_values(
    ["window", "fleet_size"]
)

fleet_figure = go.Figure()
for window, colour in zip(
    PRODUCT_WINDOWS,
    (SERIES_COLOURS["difference"], SERIES_COLOURS["selected"], SERIES_COLOURS["normal"]),
):
    window_rows = largest_curve.loc[largest_curve["window"] == window]
    fleet_figure.add_trace(
        go.Scatter(
            x=window_rows["fleet_size"],
            y=window_rows["firm_share"],
            mode="lines+markers",
            line={"color": colour},
            name=f"{window} (P10 ÷ P50)",
        )
    )
fleet_figure.update_layout(
    title=f"Firm share of 1-h turn-down by window, sub-fleets of the {max(runs)}-EV run",
    xaxis_title="Fleet size (EVs)",
    yaxis_title="Firm share (P10 ÷ P50)",
    yaxis_range=[0, 1],
)
apply_notebook_style(fleet_figure, height=380)
fleet_figure.show()
fleet_curve[
    [
        "run_vehicles",
        "fleet_size",
        "window",
        "p10_kw",
        "p50_kw",
        "firm_share",
        "implied_rho",
        "n_eff",
    ]
].round(3)

,run_vehicles,fleet_size,window,p10_kw,p50_kw,firm_share,implied_rho,n_eff
1,300,100,evening,1.211,5.777,0.210,0.381,2.580
9,300,100,overnight,33.663,46.590,0.723,0.246,3.941
17,300,100,morning,0.000,0.088,0.000,0.126,7.174
25,300,300,evening,4.963,17.410,0.285,0.317,3.133
33,300,300,overnight,95.295,130.091,0.733,0.226,4.382
41,300,300,morning,0.000,0.348,0.000,0.132,7.316


## 9. Sanity checks

Each check can fail. A failure stops the notebook here.

- The world split is disjoint and complete, and no target or intraday column is a feature.
- The bottom-up rule used in (a), applied to *all* weeks, reproduces the run's own
  `availability_bands` P10, P50 and P90 for the target: the notebook uses the model's rule, not a
  look-alike.
- For both variants, the conformal band is never narrower than the raw band (`Q ≥ 0`; section 4 prints why `Q` is
  what it is in this run), and its test coverage is roughly 80%,
  asserted loosely (0.60–0.95) because rows within a week are dependent and there are only about
  eight test weeks (review note O7).
- `firm_share` does not fall with fleet size beyond noise: the last point is at least the first
  minus `FIRM_SHARE_NOISE` (a notebook tolerance for a P10 ÷ P50 ratio estimated from 40 weeks,
  not a model assumption). With one fleet size the check would be trivially true.
- The price feature uses no price published after the night's day-ahead decision: every price
  published after `τ_DA(n)` is replaced by random numbers and night `n`'s ranks must not move.
- Shapes and NaN rules: the history has one row per (fleet, week, slot), no missing feature, and
  the intraday band is present exactly from the intraday decision half-hour onwards.

In [15]:
FIRM_SHARE_NOISE = 0.1
checks: dict[str, bool] = {}

all_worlds = np.sort(base.world_nights["world_id"].unique())
parts = [set(worlds.tolist()) for worlds in split_worlds.values()]
checks["split is disjoint and complete"] = sum(len(part) for part in parts) == len(
    all_worlds
) and set().union(*parts) == set(all_worlds)
checks["no realised quantity of the night is a feature"] = not set(ALL_FEATURES) & {
    TARGET,
    *INTRADAY_COLUMNS,
    "deliverable_known_kw",
    "deliverable_late_kw",
    "potential_kw",
    "eligible_power_kw",
}

# (a)'s rule on every week against the run's own bands.
all_week_target = base.availability_world_slot.loc[
    (base.availability_world_slot["direction"] == TARGET_DIRECTION)
    & (base.availability_world_slot["duration_hours"] == TARGET_DURATION_HOURS)
].pivot(index="slot_index", columns="world_id", values=TARGET)
run_bands = base.availability_bands.loc[
    (base.availability_bands["horizon"] == "day_ahead")
    & (base.availability_bands["direction"] == TARGET_DIRECTION)
    & (base.availability_bands["duration_hours"] == TARGET_DURATION_HOURS)
].set_index("slot_index")
recomputed = np.quantile(all_week_target.to_numpy(), LEVELS, axis=1, method="linear").T
stored = run_bands.loc[all_week_target.index, ["p10", "p50", "p90"]].to_numpy()
checks["bottom-up rule reproduces availability_bands"] = bool(
    np.allclose(recomputed, stored, rtol=1e-9, atol=1e-9, equal_nan=True)
)

checks["conformal band never narrower than raw"] = all(
    (history[f"{v.replace('top_down', 'conformal')}_p10"] <= history[f"{v}_p10"]).all()
    and (history[f"{v.replace('top_down', 'conformal')}_p90"] >= history[f"{v}_p90"]).all()
    for v in VARIANTS
)
checks["conformal test coverage in 0.60-0.95"] = all(
    0.60 <= test[TARGET].between(test[f"{c}_p10"], test[f"{c}_p90"]).mean() <= 0.95
    for c in ("conformal", "conformal_price")
)

firm_ok = True
for (_, _), rows in fleet_curve.groupby(["run_vehicles", "window"]):
    ordered = rows.sort_values("fleet_size")["firm_share"].dropna()
    if len(ordered) > 1:
        firm_ok &= bool(ordered.iloc[-1] >= ordered.iloc[0] - FIRM_SHARE_NOISE)
checks["firm share does not fall with fleet size beyond noise"] = firm_ok

checks["one row per (fleet, week, slot)"] = not history.duplicated(
    ["enrolled_evs", "world_id", "slot_index"]
).any()
checks["no missing feature"] = bool(history[ALL_FEATURES].notna().all().all())
# The decision half-hour's noon-to-noon position (17:00 is 10 at the default record).
decision_hour, decision_minute = map(int, INTRADAY_DECISION_TIME.split(":"))
decision_order = (2 * decision_hour + decision_minute // 30 - 24) % 48
from_decision = history["profile_order"] >= decision_order
checks["intraday band present exactly from the decision half-hour"] = bool(
    history.loc[from_decision, "intraday_p50"].notna().all()
    and history.loc[~from_decision, "intraday_p50"].isna().all()
)
# Scramble every price published after each night's tau_DA; that night's ranks must not move.
scramble_rng = np.random.default_rng(SEED)
price_cut_off_ok = True
for result in runs.values():
    prices, starts = study_day_ahead_prices(result)
    nights = result.study_slots["night_index"].to_numpy()
    publication = day_ahead_publication_utc_ns(starts)
    for night in np.unique(nights[nights >= 2]):
        in_night = nights == night
        ranks, decision_ns = night_visible_rank(prices, starts, in_night)
        scrambled = prices.copy()
        late = publication > decision_ns
        scrambled[:, late] = scramble_rng.uniform(-500.0, 500.0, size=(len(prices), late.sum()))
        price_cut_off_ok &= bool(
            late[in_night].any()  # the night does reach past the decision's published prices
            and np.array_equal(ranks, night_visible_rank(scrambled, starts, in_night)[0])
        )
checks["price feature uses no price published after tau_DA"] = price_cut_off_ok
checks["results are labelled synthetic"] = all(
    result.evidence_kind == "illustrative_synthetic" for result in runs.values()
)

failed = [name for name, passed in checks.items() if not passed]
assert not failed, failed
print(f"Notebook run time: {time.perf_counter() - NOTEBOOK_START:.0f} s")
pd.Series(checks, name="passed").to_frame()

Notebook run time: 50 s


,passed
split is disjoint and complete,True
no realised quantity of the night is a feature,True
bottom-up rule reproduces availability_bands,True
conformal band never narrower than raw,True
conformal test coverage in 0.60-0.95,True
firm share does not fall with fleet size beyond noise,True
"one row per (fleet, week, slot)",True
no missing feature,True
intraday band present exactly from the decision half-hour,True
price feature uses no price published after tau_DA,True


## 10. Limitations

- **Synthetic history fitted on the model's own output.** Both forecasts are fitted to, and scored
  on, simulated weeks. The comparison tests two estimators on the same synthetic world, not either
  one against reality. Real telemetry would bring things the simulation does not have: meter
  errors, changing enrolment, drift in driver habits.
- **The top-down model cannot see a maker outage coming.** An outage on a test night is drawn
  fresh; none of the day-ahead features hints at it, so it shows up only as a miss below the band.
  The bottom-up band carries outages only as the share of training weeks that had one.
- **The weather features are a perfect forecast.** `temperature_c` (and `solar_clearness` where
  the price generator provides it) are the sampled values, not a forecast with error, so the
  top-down model is flattered on any week whose weather matters.
- **The intraday band is not a day-ahead competitor.** It uses the driveway at the intraday
  decision time, hours after the day-ahead cut-off, and covers only the rest of the night from
  then; its scores sit beside the others for context, not as a ranking.
- **Small and dependent test set.** Eight test weeks, with strongly dependent rows inside each, so
  every coverage figure here is uncertain by several percentage points; the conformal guarantee
  is for exchangeable rows and holds only approximately.
- **The pinball comparison is uncertain too.** The ranking of (a), (d) and (d′) rests on eight
  paired weeks; section 5 prints the weekly differences with their standard error, and a gap
  within about two standard errors is not evidence that one method is better. Another seed or
  split could reorder them.
- **A zero-inflated target.** At this fleet size the 1-h turn-down is exactly 0 kW in most
  window half-hours, so P10 (and often P50) is 0 kW for every method and coverage below P10 is
  trivially 0. The comparison is really about the upper tail and the median. A larger fleet
  (the app's default is 1,000 EVs) or a shorter duration gives a less lumpy target.
- **The price feature is half a forecast.** Before midnight it ranks published prices; after
  midnight it ranks the expected shape, because those prices are not out at 13:00 on the day
  before. And the shape here averages the published study days only (the result carries no
  warm-up prices), not the planner's seven days, so after midnight it is close to, not identical
  with, what the planner ranked.
- **Fleet size is barely a feature.** With one fleet, `enrolled_evs` is a constant the model
  ignores; the optional second run shows two sizes. A top-down model asked about a fleet
  size it never saw would be extrapolating.